# Evaluation 

In [ ]:
from pathlib import Path
import os
import sys

# Set this to your project directory, or set the SEMEXTRACT_PROJECT environment variable.
# None discovers the project from the current working directory or its parents.
PROJECT_OVERRIDE = os.environ.get("SEMEXTRACT_PROJECT")  # e.g. "/path/to/multimodal_extraction"

if PROJECT_OVERRIDE:
    PROJECT = Path(PROJECT_OVERRIDE).expanduser().resolve()
else:
    PROJECT = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "semextract_revised").is_dir()),
        None,
    )
if PROJECT is None or not (PROJECT / "semextract_revised").is_dir():
    raise FileNotFoundError("Set PROJECT_OVERRIDE to your multimodal_extraction directory.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))


def project_path(value):
    """Resolve an absolute path, or a path relative to PROJECT; expand ~."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else PROJECT / path).resolve()


print("Project directory:", PROJECT)


In [ ]:
import glob
import importlib
from collections import Counter

import pandas as pd
from IPython.display import display

import semextract_revised.evaluation as ev
from semextract_revised.common import read_json

importlib.invalidate_caches()
importlib.reload(ev)


In [ ]:
# ---------------------------------------------------------------- settings
SPLIT_CONFIG = {
    "val": {
        "gt_path": "gt/merged_validation_fixed.json",
        "dataset_dirs": {
            "pubmed": "../code/caise27_data/val/pubmed",
        },
    },
    "test": {
        "gt_path": "gt/annotations_combined_val.json",
        "dataset_dirs": {
            "pubmed": "../code/caise27_data/test/pubmed_test",
        },
    },
}

# Papers not found in any configured PDF folder.
# Set to None to leave them unassigned.
DEFAULT_DATASET = "bo8"

# Resolve all input paths relative to PROJECT (absolute paths also work).
for config in SPLIT_CONFIG.values():
    config["gt_path"] = project_path(config["gt_path"])
    config["dataset_dirs"] = {
        name: project_path(pattern) for name, pattern in config["dataset_dirs"].items()
    }

SPLIT = "test"
if SPLIT not in SPLIT_CONFIG:
    raise ValueError("SPLIT must be 'val' or 'test'.")

# Set RUN_DIR to the STAGE_DIR printed by the extraction notebook.
RUN_DIR = project_path(os.environ.get("SEMEXTRACT_RUN_DIR", "out/stages/vllm_test/2026-11-10_12-52"))
CONTEXTS_PATH = RUN_DIR / "contexts.json"  # used only for relaxed name matching
REPORT_DIR = project_path(os.environ.get("SEMEXTRACT_REPORT_DIR", str(RUN_DIR / "metrics")))

# Preserve the original notebook's separate matching choices.
# Set both to the same mode if you want directly comparable scores and tests.
PERFORMANCE_NAME_MATCHING = "exact"
SIGNIFICANCE_NAME_MATCHING = "relaxed"

# Explicit prediction filenames; metadata JSON files are never evaluated.
FILES = [
    "early_filtered_separate_before_revision.json",
    "early_filtered_separate_after_revision.json",
    "text_filtered_separate_with_tables_before_revision.json",
    "text_filtered_separate_with_tables_after_revision.json",
    "image_then_text_entire_in_text_before_revision.json",
    "image_then_text_entire_in_text_after_revision.json",
    "images_before_revision.json",
    "images_after_revision.json",
    "late_llm_filtered_in_text_before_revision.json",
    "late_llm_filtered_in_text_after_revision.json",
    "text_then_image_filtered_in_text_before_revision.json",
    "text_then_image_filtered_in_text_after_revision.json",
]

if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Prediction directory not found: {RUN_DIR}. Set RUN_DIR above.")
if not SPLIT_CONFIG[SPLIT]["gt_path"].is_file():
    raise FileNotFoundError(f"Ground truth not found: {SPLIT_CONFIG[SPLIT]['gt_path']}. Edit SPLIT_CONFIG.")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ("performance_reports", "revision_reports", "pairwise_reports"):
    (REPORT_DIR / folder).mkdir(parents=True, exist_ok=True)
print("Predictions:", RUN_DIR)
print("Reports:", REPORT_DIR)


In [ ]:
def contexts_for(name_matching):
    """Only relaxed matching needs the run's saved extraction contexts."""
    if name_matching != "relaxed":
        return None
    if not CONTEXTS_PATH.is_file():
        raise FileNotFoundError(f"Contexts not found: {CONTEXTS_PATH}. Edit CONTEXTS_PATH.")
    return read_json(CONTEXTS_PATH)


def find_pdf_datasets(dataset_dirs):
    """Map each PDF stem, lowercased, to its dataset(s)."""
    file_ids, folders = {}, Counter()

    for dataset, pattern in dataset_dirs.items():
        for directory in sorted(glob.glob(str(pattern))):
            directory_path = Path(directory)
            if not directory_path.is_dir():
                continue

            pdfs = [
                p for p in directory_path.iterdir()
                if p.is_file() and p.suffix.lower() == ".pdf"
            ]
            folders[str(directory_path)] = len(pdfs)

            for pdf in pdfs:
                file_ids.setdefault(pdf.stem.casefold(), set()).add(dataset)

    return file_ids, folders

def evaluate_predictions(
    pred_path,
    *,
    split,
    name_matching="relaxed",
    contexts=None,
    name_map=None,
    default_dataset=DEFAULT_DATASET,
    verbose=False,
):
    """Evaluate predictions and display scores per dataset.

    name_matching:
        "exact", "normalized", "aliases", or "relaxed".

    contexts:
        Prediction contexts dictionary or JSON path.
        Only supported with name_matching="relaxed".

    name_map:
        Manual mapping dictionary or JSON path:
        {article_id: {predicted_name: gold_name}}.
        Only supported with name_matching="aliases".

    Returns:
        A pandas DataFrame containing scores per dataset and tuple size.
    """
    if split not in SPLIT_CONFIG:
        raise ValueError("split must be 'val' or 'test'")

    if contexts is not None and name_matching != "relaxed":
        raise ValueError(
            "contexts is only supported with name_matching='relaxed'"
        )

    if name_map is not None and name_matching != "aliases":
        raise ValueError(
            "Manual mappings require name_matching='aliases'"
        )

    config = SPLIT_CONFIG[split]
    gt_path = config["gt_path"]
    dataset_dirs = config["dataset_dirs"]

    print(f"[{split}] {pred_path}")

    # ---- evaluate and save the full per-paper report
    result = ev.evaluate(
        gt_path=gt_path,
        predictions_path=pred_path,
        output_path=REPORT_DIR / "performance_reports" / f"{Path(pred_path).stem}.json",
        split=split,
        name_matching=name_matching,
        contexts=contexts,
        name_map=name_map,
    )

    if verbose:
        print("Matching:", result["settings"]["matching"])
        print(
            "Invalid relations:",
            {
                "gold": result["invalid_gold_relations"],
                "predictions": result["invalid_prediction_relations"],
            },
        )
        print(
            "Articles without predictions:",
            len(result["articles_without_predictions"]),
        )

    # ---- relaxed name-matching diagnostics
    if verbose and name_matching == "relaxed":
        matches = pd.DataFrame(
            [
                {
                    "article_id": article["article_id"],
                    "predicted_name": predicted,
                    "gold_name": gold,
                }
                for article in result["articles"]
                for predicted, gold
                in article["relaxed_name_mapping"].items()
            ],
            columns=["article_id", "predicted_name", "gold_name"],
        )

        identical = matches["predicted_name"].eq(matches["gold_name"])
        similarity_matches = matches.loc[~identical].copy()

        print("Total matched variables:", len(matches))
        print(
            "Identical after relaxed normalization:",
            int(identical.sum()),
        )
        print("Matched through similarity:", len(similarity_matches))
        print("Stemming:", result["settings"]["relaxed_stemming"])

        if not similarity_matches.empty:
            display(
                similarity_matches
                .sort_values(["article_id", "predicted_name"])
                .reset_index(drop=True)
            )

    # ---- manual mapping diagnostics
    if verbose and name_map is not None:
        mapping_rows = []

        for article in result["articles"]:
            diagnostic = article["manual_mapping"]
            mapping_rows.append({
                "article_id": article["article_id"],
                "mapping_entries": diagnostic["entries"],
                "unused_entries": diagnostic["unused"],
                "many_to_one": diagnostic["many_to_one"],
            })

        print(
            "Manual mapping entries:",
            sum(row["mapping_entries"] for row in mapping_rows),
        )
        print(
            "Unused mapping entries:",
            sum(len(row["unused_entries"]) for row in mapping_rows),
        )

        noteworthy = [
            row for row in mapping_rows
            if row["unused_entries"] or row["many_to_one"]
        ]
        if noteworthy:
            display(pd.DataFrame(noteworthy))

    # ---- identify each paper's dataset from the PDF folders
    file_ids, folders = find_pdf_datasets(dataset_dirs)

    if verbose:
        print(
            "PDF folders:",
            dict(folders) or "none found - check SPLIT_CONFIG",
        )

    papers, _ = ev.find_gold_papers(read_json(gt_path))
    gt_key = {
        str(entry.get("paper_id") or ev.strip_extension(key)): key
        for key, entry in papers
    }

    def identifiers(record):
        """Paper, gold and prediction IDs without extensions, case-insensitive."""
        values = {
            record["article_id"],
            gt_key.get(record["article_id"], ""),
            record.get("prediction_source") or "",
        }
        return {
            ev.strip_extension(value).casefold()
            for value in values
            if value
        }

    dataset_of, assigned = {}, Counter()

    for record in result["articles"]:
        found = set()
        for identifier in identifiers(record):
            found.update(file_ids.get(identifier, set()))

        if len(found) > 1:
            print(
                f"WARNING: {record['article_id']} found in several datasets: "
                f"{sorted(found)}"
            )

        if found:
            dataset_of[record["article_id"]] = sorted(found)[0]
            assigned["from folder"] += 1
        elif default_dataset:
            dataset_of[record["article_id"]] = default_dataset
            assigned["default"] += 1
        else:
            assigned["unassigned"] += 1

    if verbose:
        print(
            "Assignment:",
            dict(assigned),
            "| per dataset:",
            dict(Counter(dataset_of.values())),
        )

        if not assigned["from folder"]:
            print(
                "No paper matched a PDF name. Compare, e.g.:",
                sorted(file_ids)[:3],
                "vs",
                [r["article_id"] for r in result["articles"][:3]],
            )

    # ---- scores per dataset
    groups = {"all": result["articles"]}

    for name in sorted(set(dataset_of.values())):
        groups[name] = [
            article
            for article in result["articles"]
            if dataset_of.get(article["article_id"]) == name
        ]

    rows = []

    for name, records in groups.items():
        if not records:
            continue

        summary = ev.summarize(records)

        for size in ev.SCORE_TYPES:
            micro = summary[size]["micro"]
            macro = summary[size]["macro"]

            rows.append({
                "split": split,
                "dataset": name,
                "size": size,
                "papers": len(records),
                "gold": micro["gold"],
                "predicted": micro["predicted"],
                "tp": micro["tp"],
                "micro_P": micro["precision"],
                "micro_R": micro["recall"],
                "micro_F1": micro["f1"],
                "macro_P": macro["precision"],
                "macro_R": macro["recall"],
                "macro_F1": macro["f1"],
            })

    scores = pd.DataFrame(rows)
    display(scores.round(3))
    return scores


In [ ]:
# Evaluate the configured files, then save the combined table at full precision.
if not FILES:
    raise ValueError("Set FILES to the prediction filenames you want to evaluate.")
missing = [str(RUN_DIR / name) for name in FILES if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Missing prediction files:\n" + "\n".join(missing))

contexts = contexts_for(PERFORMANCE_NAME_MATCHING)
all_scores = []
for fname in FILES:
    scores = evaluate_predictions(
        RUN_DIR / fname,
        split=SPLIT,
        name_matching=PERFORMANCE_NAME_MATCHING,
        contexts=contexts,
        name_map=None,
        verbose=False,
    )
    all_scores.append(scores.assign(file=fname))

results = pd.concat(all_scores, ignore_index=True)
output_path = REPORT_DIR / "extraction_performance.csv"
results.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

# ---- top 5 for entities and each tuple size, across all papers
overall = results.loc[results["dataset"] == "all"]

for metric in ["micro_F1", "macro_F1"]:
    prefix = metric.split("_")[0]

    for size in ev.SCORE_TYPES:  # entities, 3, 4, 5
        candidates = overall.loc[overall["size"] == size]
        if candidates.empty:
            continue

        top = candidates.nlargest(5, metric)
        label = "entities" if size == "entities" else f"{size}-tuples"

        print(f"Top 5 by {metric}: {label}")
        display(
            top[
                ["file", "gold", "predicted", "tp",
                 metric, f"{prefix}_P", f"{prefix}_R"]
            ].round(3)
        )


## Before/after revision significance
Run cells 2–5 first. Edit `METHODS` below to match the before/after files in
`RUN_DIR`. This saves `revision_significance_micro_f1.csv` and per-file JSON reports
under `REPORT_DIR`, using `SIGNIFICANCE_NAME_MATCHING`.


In [ ]:
METHODS = [
    "early_filtered_separate",
    "text_filtered_separate_with_tables",
    "image_then_text_entire_in_text",
    "images",
    "late_llm_filtered_in_text",
    "text_then_image_filtered_in_text",
]

PAIRS = [
    (
        f"{method}_before_revision.json",
        f"{method}_after_revision.json",
    )
    for method in METHODS
]

SCORE_TYPES = ["3", "4", "5"]
AVERAGING = ("micro",)

N_RESAMPLES = 10_000
SEED = 42
ALPHA = 0.05

run_dir = Path(RUN_DIR)
config = SPLIT_CONFIG[SPLIT]

# Use the configured matching mode and the same contexts for every file.
evaluation_contexts = contexts_for(SIGNIFICANCE_NAME_MATCHING)

def holm_adjust(pvalues):
    """Adjust one family of p-values using Holm's method."""
    m = len(pvalues)
    adjusted = [0.0] * m
    running = 0.0

    for rank, index in enumerate(
        sorted(range(m), key=pvalues.__getitem__)
    ):
        running = max(running, (m - rank) * pvalues[index])
        adjusted[index] = min(1.0, running)

    return adjusted


# Check that all before/after files exist.
filenames = sorted({name for pair in PAIRS for name in pair})

missing = [
    str(run_dir / name)
    for name in filenames
    if not (run_dir / name).is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing prediction files:\n" + "\n".join(missing)
    )


# Retain full evaluation reports, including per-paper counts.
reports = {}

for fname in filenames:
    pred_path = run_dir / fname
    print(f"Evaluating {fname}")

    reports[fname] = ev.evaluate(
        gt_path=config["gt_path"],
        predictions_path=pred_path,
        output_path=REPORT_DIR / "revision_reports" / f"{pred_path.stem}.json",
        split=SPLIT,
        name_matching=SIGNIFICANCE_NAME_MATCHING,
        contexts=evaluation_contexts,
        name_map=None,
    )


# Paired comparisons: after minus before.
rows = []

for method, (before_file, after_file) in zip(METHODS, PAIRS):
    before = reports[before_file]
    after = reports[after_file]

    for size in SCORE_TYPES:
        print(f"Testing revision: {method}, {size}-tuple")

        comparison = ev.compare_reports(
            before,
            after,
            size=size,
            samples=N_RESAMPLES,
            seed=SEED,
        )

        for averaging in AVERAGING:
            statistics = comparison["results"][averaging]
            ci_low, ci_high = statistics["bootstrap_95_ci"]

            rows.append({
                "method": method,
                "dataset": "all",
                "size": size,
                "metric": f"{averaging}_F1",
                "papers": comparison["articles"],
                "before_F1": before["summary"][size][averaging]["f1"],
                "after_F1": after["summary"][size][averaging]["f1"],
                "delta_F1": statistics["difference_second_minus_first"],
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "p_raw": statistics["permutation_p_two_sided"],
            })

revision_significance = pd.DataFrame(rows)

# Holm correction across all configured comparisons.
# Changing SCORE_TYPES or AVERAGING changes this correction family.
revision_significance["p_holm"] = holm_adjust(
    revision_significance["p_raw"].tolist()
)

revision_significance["significant"] = revision_significance["p_holm"] < ALPHA

revision_significance["conclusion"] = [
    (
        "significant increase" if delta > 0
        else "significant decrease"
    ) if significant else "not significant"
    for delta, significant in zip(
        revision_significance["delta_F1"],
        revision_significance["significant"],
    )
]

print(
    f"\n{len(revision_significance)} comparisons; "
    f"Holm correction across all comparisons; alpha={ALPHA}."
)
print(
    "Positive delta_F1 favors revision. "
    "Bootstrap confidence intervals are unadjusted."
)

display(revision_significance.round(6))

# Save full-precision results.
output_path = REPORT_DIR / "revision_significance_micro_f1.csv"
revision_significance.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


## Pairwise method significance
Run cells 2–5 first. Edit `PAIRS` below to match the methods you want to compare.
This saves `pairwise_significance_f1.csv` and per-file JSON reports under
`REPORT_DIR`, using `SIGNIFICANCE_NAME_MATCHING`. The revision test is not required.


In [ ]:
# Each pair is (baseline, candidate).
# Choose prediction filenames in RUN_DIR; these may differ from FILES above.
PAIRS = [
    ("early_filtered_separate.json", "text_filtered_separate_with_tables.json"),
    ("early_filtered_separate.json", "image_then_text_entire_in_text_before_revision.json"),
    ("early_filtered_separate.json", "images.json"),
    ("early_filtered_separate.json", "late_llm_filtered_in_text.json"),
    ("early_filtered_separate.json", "text_then_image_filtered_in_text.json"),
    ("early_filtered_separate.json", "baseline.json"),
    ("text_filtered_separate_with_tables.json", "image_then_text_entire_in_text_before_revision.json"),
    ("text_filtered_separate_with_tables.json", "images.json"),
    ("text_filtered_separate_with_tables.json", "late_llm_filtered_in_text.json"),
    ("text_filtered_separate_with_tables.json", "text_then_image_filtered_in_text.json"),
    ("text_filtered_separate_with_tables.json", "baseline.json"),
    ("image_then_text_entire_in_text_before_revision.json", "images.json"),
    ("image_then_text_entire_in_text_before_revision.json", "late_llm_filtered_in_text.json"),
    ("image_then_text_entire_in_text_before_revision.json", "text_then_image_filtered_in_text.json"),
    ("image_then_text_entire_in_text_before_revision.json", "baseline.json"),
    ("images.json", "late_llm_filtered_in_text.json"),
    ("images.json", "text_then_image_filtered_in_text.json"),
    ("images.json", "baseline.json"),
    ("late_llm_filtered_in_text.json", "text_then_image_filtered_in_text.json"),
    ("late_llm_filtered_in_text.json", "baseline.json"),
    ("text_then_image_filtered_in_text.json", "baseline.json"),
]

N_PERMUTATIONS = 10_000
SEED = 42
ALPHA = 0.05


def holm_adjust(pvalues):
    """Adjust one family of p-values using Holm's method."""
    m = len(pvalues)
    adjusted = [0.0] * m
    running = 0.0

    for rank, index in enumerate(
        sorted(range(m), key=pvalues.__getitem__)
    ):
        running = max(running, (m - rank) * pvalues[index])
        adjusted[index] = min(1.0, running)

    return adjusted


# Keep the full reports: aggregate F1 scores alone are insufficient.
evaluation_contexts = contexts_for(SIGNIFICANCE_NAME_MATCHING)
reports = {}

for fname in sorted({name for pair in PAIRS for name in pair}):
    pred_path = RUN_DIR / fname
    if not pred_path.is_file():
        raise FileNotFoundError(pred_path)

    print(f"Evaluating {fname}")
    reports[fname] = ev.evaluate(
        gt_path=SPLIT_CONFIG[SPLIT]["gt_path"],
        predictions_path=pred_path,
        output_path=REPORT_DIR / "pairwise_reports" / f"{pred_path.stem}.json",
        split=SPLIT,
        name_matching=SIGNIFICANCE_NAME_MATCHING,
        contexts=evaluation_contexts,
        name_map=None,
    )

rows = []

for baseline, candidate in PAIRS:
    first = reports[baseline]
    second = reports[candidate]

    for size in ev.SCORE_TYPES:  # entities, 3, 4, 5
        print(f"Comparing {baseline} vs {candidate}: {size}")

        comparison = ev.compare_reports(
            first,
            second,
            size=size,
            samples=N_PERMUTATIONS,
            seed=SEED,
        )

        for averaging in ("micro", "macro"):
            statistics = comparison["results"][averaging]

            rows.append({
                "baseline": baseline,
                "candidate": candidate,
                "dataset": "all",
                "size": size,
                "metric": f"{averaging}_F1",
                "papers": comparison["articles"],
                "baseline_F1": first["summary"][size][averaging]["f1"],
                "candidate_F1": second["summary"][size][averaging]["f1"],
                "delta_F1": statistics["difference_second_minus_first"],
                "p_raw": statistics["permutation_p_two_sided"],
            })

pairwise_significance = pd.DataFrame(rows)

# One correction family: all pairs × score types × averaging methods above.
pairwise_significance["p_holm"] = holm_adjust(pairwise_significance["p_raw"].tolist())
pairwise_significance["significant"] = pairwise_significance["p_holm"] < ALPHA

display(pairwise_significance.round(6))
output_path = REPORT_DIR / "pairwise_significance_f1.csv"
pairwise_significance.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
